In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             confusion_matrix,
                             roc_auc_score,
                             roc_curve,
                             precision_recall_curve,
                             average_precision_score,
                             f1_score)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import (StratifiedKFold,
                                     cross_val_score)
import joblib
import warnings
warnings.filterwarnings('ignore')

SAVE_PATH = "/content/drive/MyDrive/final_project/"
BANK_PATH = "/content/drive/MyDrive/project/bank-additional/bank-additional-full.csv"

print("Phase 3 imports done ✅")

Phase 3 imports done ✅


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv(BANK_PATH, sep=";")

print(f"Shape    : {df.shape}")
print(f"\nColumns  : {df.columns.tolist()}")
print(f"\nTarget distribution:")
print(df['y'].value_counts())
print(f"\nClass balance:")
print(df['y'].value_counts(normalize=True).round(3))
print(f"\nMissing values:")
print(df.isnull().sum().sum())

Shape    : (41188, 21)

Columns  : ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']

Target distribution:
y
no     36548
yes     4640
Name: count, dtype: int64

Class balance:
y
no     0.887
yes    0.113
Name: proportion, dtype: float64

Missing values:
0


In [ ]:
import numpy as np

df_clean = df.copy()

# Encode target
df_clean['y'] = (df_clean['y'] == 'yes').astype(int)

# Drop duration — causes data leakage
# duration is only known AFTER call ends
# not available before making decision
df_clean.drop('duration', axis=1, inplace=True)

# Identify column types
cat_cols = df_clean.select_dtypes(
    include='object'
).columns.tolist()
num_cols = df_clean.select_dtypes(
    include=np.number
).columns.tolist()
num_cols.remove('y')

print(f"Categorical : {cat_cols}")
print(f"Numerical   : {num_cols}")

# One-Hot Encoding
df_encoded = pd.get_dummies(
    df_clean,
    columns=cat_cols,
    drop_first=True,
    dtype=int
)

X = df_encoded.drop('y', axis=1)
y = df_encoded['y']

print(f"\nShape before OHE : {df_clean.shape}")
print(f"Shape after OHE  : {df_encoded.shape}")
print(f"Final X shape    : {X.shape}")
print(f"Positive class   : {y.mean()*100:.1f}%")

Categorical : ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']
Numerical   : ['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

Shape before OHE : (41188, 20)
Shape after OHE  : (41188, 53)
Final X shape    : (41188, 52)
Positive class   : 11.3%


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.1,
    random_state=42,
    stratify=y_train
)

print(f"Train : {X_train.shape}")
print(f"Val   : {X_val.shape}")
print(f"Test  : {X_test.shape}")

# Class weights
weights = compute_class_weight(
    'balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(weights))
scale_pos = float(weights[1] / weights[0])

print(f"\nClass weights : {class_weight_dict}")
print(f"Scale pos     : {scale_pos:.4f}")

Train : (29655, 52)
Val   : (3295, 52)
Test  : (8238, 52)

Class weights : {0: np.float64(0.5634833168655469), 1: np.float64(4.438042502244837)}
Scale pos     : 7.8761


In [ ]:
xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos,
    random_state=42,
    eval_metric='auc',
    early_stopping_rounds=30,
    verbosity=0
)

xgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

xgb_prob = xgb.predict_proba(X_test)[:,1]
print(f"XGBoost AUC : {roc_auc_score(y_test, xgb_prob):.4f}")

XGBoost AUC : 0.8164


In [ ]:
import lightgbm as lgb_module

lgbm = LGBMClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight='balanced',
    random_state=42,
    verbose=-1
)

lgbm.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb_module.early_stopping(30, verbose=False),
        lgb_module.log_evaluation(False)
    ]
)

lgbm_prob = lgbm.predict_proba(X_test)[:,1]
print(f"LightGBM AUC : {roc_auc_score(y_test, lgbm_prob):.4f}")

LightGBM AUC : 0.8034


In [ ]:
catboost = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    eval_metric='AUC',
    early_stopping_rounds=30,
    class_weights=class_weight_dict,
    random_seed=42,
    verbose=0
)

catboost.fit(
    X_train, y_train,
    eval_set=(X_val, y_val)
)

cat_prob = catboost.predict_proba(X_test)[:,1]
print(f"CatBoost AUC : {roc_auc_score(y_test, cat_prob):.4f}")

CatBoost AUC : 0.8110


In [ ]:
# Soft voting ensemble
ensemble_prob = (xgb_prob + lgbm_prob + cat_prob) / 3
print(f"Ensemble AUC : {roc_auc_score(y_test, ensemble_prob):.4f}")

# Threshold optimization on val set
val_xgb  = xgb.predict_proba(X_val)[:,1]
val_lgbm = lgbm.predict_proba(X_val)[:,1]
val_cat  = catboost.predict_proba(X_val)[:,1]
val_ens  = (val_xgb + val_lgbm + val_cat) / 3

thresholds = np.arange(0.1, 0.9, 0.01)
f1_scores  = []

for t in thresholds:
    preds = (val_ens >= t).astype(int)
    f1_scores.append(
        f1_score(y_val, preds, zero_division=0)
    )

best_threshold = thresholds[np.argmax(f1_scores)]
print(f"Best threshold : {best_threshold:.2f}")

ensemble_pred = (
    ensemble_prob >= best_threshold
).astype(int)

print(f"\nFinal Accuracy : "
      f"{accuracy_score(y_test, ensemble_pred)*100:.2f}%")
print(f"\nClassification Report:")
print(classification_report(
    y_test, ensemble_pred,
    target_names=['Not Subscribed', 'Subscribed']
))

Ensemble AUC : 0.8129
Best threshold : 0.71

Final Accuracy : 89.06%

Classification Report:
                precision    recall  f1-score   support

Not Subscribed       0.94      0.93      0.94      7310
    Subscribed       0.51      0.55      0.53       928

      accuracy                           0.89      8238
     macro avg       0.73      0.74      0.73      8238
  weighted avg       0.89      0.89      0.89      8238



In [ ]:
cv = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=42
)

cv_models = {
    'XGBoost' : XGBClassifier(
        n_estimators=300, max_depth=6,
        learning_rate=0.05,
        scale_pos_weight=scale_pos,
        random_state=42, verbosity=0,
        eval_metric='auc'
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=300, max_depth=6,
        learning_rate=0.05,
        class_weight='balanced',
        random_state=42, verbose=-1
    )
}

cv_results = {}

print("Running 5-Fold CV...")
for name, clf in cv_models.items():
    scores = cross_val_score(
        clf, X, y,
        cv=cv,
        scoring='roc_auc',
        n_jobs=-1
    )
    cv_results[name] = scores
    print(f"\n{name}")
    print(f"  Fold AUCs : {np.round(scores, 4)}")
    print(f"  Mean AUC  : {scores.mean():.4f} "
          f"± {scores.std():.4f}")

# Manual CV for CatBoost
print("\nCatBoost (Manual CV)")
cat_cv_scores = []
for fold, (tr_idx, val_idx) in enumerate(
        cv.split(X, y)):
    X_cv_tr  = X.iloc[tr_idx]
    X_cv_val = X.iloc[val_idx]
    y_cv_tr  = y.iloc[tr_idx]
    y_cv_val = y.iloc[val_idx]

    cat_cv = CatBoostClassifier(
        iterations=300, depth=6,
        learning_rate=0.05,
        class_weights=class_weight_dict,
        random_seed=42, verbose=0
    )
    cat_cv.fit(X_cv_tr, y_cv_tr)
    score = roc_auc_score(
        y_cv_val,
        cat_cv.predict_proba(X_cv_val)[:,1]
    )
    cat_cv_scores.append(score)
    print(f"  Fold {fold+1} AUC : {score:.4f}")

cat_cv_scores      = np.array(cat_cv_scores)
cv_results['CatBoost'] = cat_cv_scores
print(f"  Mean AUC  : {cat_cv_scores.mean():.4f} "
      f"± {cat_cv_scores.std():.4f}")

Running 5-Fold CV...

XGBoost
  Fold AUCs : [0.7898 0.7864 0.8089 0.7946 0.7958]
  Mean AUC  : 0.7951 ± 0.0077

LightGBM
  Fold AUCs : [0.7927 0.7963 0.8099 0.8019 0.804 ]
  Mean AUC  : 0.8010 ± 0.0060

CatBoost (Manual CV)
  Fold 1 AUC : 0.7997
  Fold 2 AUC : 0.7980
  Fold 3 AUC : 0.8154
  Fold 4 AUC : 0.7997
  Fold 5 AUC : 0.8034
  Mean AUC  : 0.8032 ± 0.0063


In [ ]:
SAVE_PATH = "/content/drive/MyDrive/final_project/"

# Save models
joblib.dump(
    xgb,
    SAVE_PATH + "behavioral_xgboost.pkl"
)
joblib.dump(
    lgbm,
    SAVE_PATH + "behavioral_lightgbm.pkl"
)
catboost.save_model(
    SAVE_PATH + "behavioral_catboost.cbm"
)

# Save full predictions for Phase 4
X_full        = df_encoded.drop('y', axis=1)
xgb_full      = xgb.predict_proba(X_full)[:,1]
lgbm_full     = lgbm.predict_proba(X_full)[:,1]
cat_full      = catboost.predict_proba(X_full)[:,1]
ensemble_full = (xgb_full + lgbm_full + cat_full) / 3

np.save(
    SAVE_PATH + "behavioral_scores.npy",
    ensemble_full
)
np.save(
    SAVE_PATH + "best_threshold.npy",
    np.array([best_threshold])
)
np.save(
    SAVE_PATH + "feature_names.npy",
    np.array(X.columns.tolist())
)

df_clean.to_csv(
    SAVE_PATH + "bank_clean.csv",
    index=False
)

print("All Phase 3 files saved ✅")
print(f"\nSaved:")
print("  behavioral_xgboost.pkl")
print("  behavioral_lightgbm.pkl")
print("  behavioral_catboost.cbm")
print("  behavioral_scores.npy")
print("  best_threshold.npy")
print("  feature_names.npy")
print("  bank_clean.csv")

All Phase 3 files saved ✅

Saved:
  behavioral_xgboost.pkl
  behavioral_lightgbm.pkl
  behavioral_catboost.cbm
  behavioral_scores.npy
  best_threshold.npy
  feature_names.npy
  bank_clean.csv
